# Data Aquisition for Bench-Top Arduino Units
## Author: Anthony (Tony, he) Butterfield, Associate Prof (Lect.) U of Utah
To be used with equipment found: https://drive.google.com/drive/folders/1KWpuyOrEbVtaMLTswfcXkQgPNCC64Uck?usp=sharing

First, we will import all the needed packages:


In [ ]:
%matplotlib qt
#The above makes it so we can make a real-time plot in a separate window in Jupyter notebooks
import matplotlib.pyplot as plt  #plot library to show our data
import serial #Import Serial Library to talk to the board
import time  #tools to keep tract of time
import pandas as pd #a sophisticated data science package... we're just going to use to save a retrieve data
import numpy as np
import serial.tools.list_ports
import os  #this was used to aid in preventing files from duplicating
from os import path
print('Imports successful....')

## Set the constants
The user may want to alter these:

In [ ]:
######################### USER CONSTANTS  ###########################
delay_time = 0.5 # seconds, approx time between each measure of voltage
max_time = 1e6 #max seconds for which data will be taken
max_data_points = 5e4

z = 0
while (path.exists("TheData-{}.csv".format(z))==True):
    z+=1
SaveFileName = 'TheData-{}.csv'.format(z)

tmp_sv_time = 120 #how long to wait before saving a temporary data file
TempFilePrefix = 'TmpData_X.csv'  #will be saved every tmp_sv_time seconds, replaceing the X with 0-9
print('Constants set...')

## Attempt communication with the arduino through the UBS port.

The baud rate should be 9600

In [ ]:
ports = list(serial.tools.list_ports.comports())  #find all the com ports
foundboard = False #test for is the board was found
for p in ports:  #loop through them and find first arduino
    print(f'Testing port: {p}')
    comportID = str(p).split(' ')[0]  #port id should be first string before first space
    print(f'Port ID: {comportID}')
    if (('COM' in comportID) or ('cu.usb' in comportID) ): #check if it's a possible pc or mac port for arduino
        try: #see if it's communication at 9600, and if so we'll treak it like an arduino
            CPSerialData = serial.Serial(comportID,9600) #Create Serial port object called CPSerialData
            foundboard = True
            print('New board connection established.')
            break #only looking for the first com port with an arduino
        except: #maybe it failed because the connection already existed...
            try:  #see if it was already connected
                serStr = CPSerialData.readline() #see if we can read from it
                if (len(serStr) > 0): #and see that it's a string in form of a dictionary
                    foundboard = True
                    print(f'Board at \'{p}\' was already connected')
                    break #only looking for the first com port with an arduino
                else: #it read something but it's not the dictionary format we expect
                    print(f'Serial com occured, but not from benchtop equipment')
            except: #it all went bad, check the cable, and that the board if found in the arduino IDE
                print(f'{p} ::: does not contain the arduino with proper code.')
if (foundboard): print('Board connected successfully')
else: print('Board not found')

Will read from the port for a bit to see what type of sensor this is and what it reads...

In [ ]:
CPSerialData.flushInput() #empty the serial buffer else you may see old v measurements
found_data = False  #when true we know we have some data
max_test_time = 20 #maximim time in seconds it will try before giving up
tic = time.time() # get the starting time
toc = 0 #will contain the time since initial tic
print('Giving it a go...')
try:#collect data until ctrl-c is used
    while (toc < max_test_time and not(found_data)): #take data as long as we aren't over the max time
        if (CPSerialData.inWaiting()>0):  #wait for the arduino to say something...
            toc = time.time() - tic  #measure the time elapsed
            print(CPSerialData.readline())
            try:#, like you really care
                serStr = CPSerialData.readline().strip().decode('ascii') #read and clean the string
                print(serStr)
                if (serStr[0] == '{'): #it's a dict.'
                    try:
                        Data = eval(serStr) #make it into a dictionary
                        found_data = True
                    except:
                        print('Found a { but it wasn\'t a dictionary')
            except:
                print('Data read error 1')
    if (toc > max_test_time):
        print('Ran out of time and found nothing of value')
    else:
        print('Found dictionary of form:')
        print(Data)
        print('Keys you may plot data from:')
        DataKeys = list(Data.keys())
        print(DataKeys)
except:
    print('Data read error 2')


If dictionary keys were found, go ahead and pick some to plot out below. All data will be recorded and saved in the csv save file.

If there is a 't' key the code will assume that's time and plot vs that. If there is not a 't' key it will make its own time vector to plot against.

In [ ]:
Keys2Plot = [DataKeys[i] for i in [0,1] ] #CHANGE THIS TO YOUR NEEDS; you could also just make a list of the key strings instead of referencing DataKeys; max number is 9
nData2Plot = len(Keys2Plot)  #how many lines will we need to plot
print('Now planning to plot only:')
print(Keys2Plot)

# Now start taking data, and plotting

In [ ]:
plt.close("all")
plt.ion() #starts interactive plot mode, so we can have an animated plot
rxc = [[1,1],[2,1],[3,1],[2,2],[3,2],[3,2],[3,3],[3,3],[3,3]] #row and column for subplot with number of data
fig, ax = plt.subplots(rxc[nData2Plot-1][0],rxc[nData2Plot-1][1])  #open a new figure in which we will plot
li = [] #list of handles to lines that will be plotted
li_xMin, li_xMax = float('inf'), float('inf')
if (nData2Plot>1): #multiple subplots
    for i in range(0,nData2Plot):
        ax[i].grid() #display a grid on the plot
        ax[i].tick_params(labelsize=14)#ticks on axis set to size 14 font
        line, = ax[i].plot( [0], [0] )
        li.append(line)
        ax[i].set_xlabel('time (sec)', fontsize=14)#x axis label with size 14 font
        ax[i].set_ylabel(Keys2Plot[i], fontsize=14)#y-axis label with size 14 font
else: #just the one plot
    ax=[ax] #make it a list with one member to standardize how ax is referenced regardless of 1 vs many plots
    ax[0].grid(); ax[0].tick_params(labelsize=14)
    line, = ax[0].plot( [0], [0] )
    li.append(line)
fig.canvas.draw()#draw the plot canvas

i=0 #variable to count the datum
count = 0
tic = time.time() # get the starting time
toc = 0 #will contain the time since initial tic
df = pd.DataFrame() #the dataframe that will hold all the variables sent from the arduino
Data = {} #where we'll keep the dictionary sent from the equipment
CPSerialData.flushInput() #empty the serial buffer else you may see old v measurements
try: #collect data until ctrl-c is used
    while toc < max_time: #take data as long as we aren't over the max time
        if (CPSerialData.inWaiting()>0):  #wait for the arduino to say something...
            toc = time.time() - tic;  #measure the time elapsed
            #try:#, like you really care
            serStr = CPSerialData.readline().strip().decode('ascii') #read and clean the string
            #print(serStr)
            if (len(serStr) > 0 and serStr[0] == '{'): #it's a dict.'
                serStr=serStr.replace("inf","float('inf')") #take care of possible infinity readings
                print(serStr) #show me what you got
                Data = eval(serStr) #make it into a dictionary
                #if (not('t' in DataKeys)):
                #    Data['t'] = toc  #if not time data is given then use computer to track time and make time = zero when code started
                Data['t'] = toc
                for key,elem in Data.items():  #loop through the variables sent back and see what's new
                    #print(key)
                    if ( not ( key in df.columns ) ):
                        df[key] = np.nan  #add a new column
                    if ( type(elem) == str ): # if it's a string then it's Not A Number, a nan
                        df.loc[i,key] = np.nan
                    else:  #then it's a number for data
                        df.loc[i,key] = Data[key]
                j = 0
                for k in Keys2Plot:
                    li[j].set_xdata(df['t']) #plot the new x data for this line
                    li[j].set_ydata(df[k]) #plot the new y data for this line
                    ax[j].set_xlim(df['t'][0], np.max(df['t']))#initial x limits
                    ax[j].set_ylim(np.min(df[k]), np.max(df[k]))#initial y limits
                    j+=1
                    print(df[k])
            print('00000000')
            fig.canvas.draw() #draw the new plot
            fig.canvas.flush_events()
            plt.pause(0.01) #pause a bit so the plot can show
            time.sleep(delay_time-0.01) #pause so we aren't taking too much data (in sec)
            i+=1 #incriment the counter
        
            CPSerialData.flushInput() #empty the serial buffer else you may see old v measurements
        if (count>=max_data_points):
            z = 0
            while (path.exists("TheData-{}.csv".format(z))==True):
                z+=1
            SaveFileName = 'TheData-{}.csv'.format(z)
            df.to_csv(SaveFileName)
            df = pd.DataFrame() #the dataframe that will hold all the variables sent from the arduino
            Data = {} #where we'll keep the dictionary sent from the equipment
            
            count = 0
            #except:
            #    print('      \Data read error/  serStr = ', serStr)
except KeyboardInterrupt:  #stop if ctrl-c is pressed
    print('Broke!!!')  #let the user know we ended the data collection
print('Done with data collection')
z = 0
while (path.exists("TheData-{}.csv".format(z))==True):
    z+=1
SaveFileName = 'TheData-{}.csv'.format(z)
df.to_csv(SaveFileName)
CPSerialData.close() #Stop communicating and let go of the com port
